In [38]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(dagbagM)')

In [2]:
#assigns node types. 'c' for continuous, 'b' for binary
def infer_type(df):
    import rpy2.robjects as ro
    node_types=[]
    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        if np.issubdtype(x.dtype, np.number):
            if len(unique_vals) == 2 and set(unique_vals).issubset({0, 1}):
                node_types.append("b")
            else:
                node_types.append("c")
        else:
            if len(unique_vals) == 2:
                node_types.append("b")
            else:
                node_types.append("c")
    return ro.StrVector(node_types)

In [31]:
def run_dagbagm(df: pd.DataFrame, seed=1):
    df_clean = df.dropna().copy()
    #infer node types on transformed df
    node_type = infer_type(df_clean)

    print("Columns used in dagbagM:", df_clean.columns.tolist())
    print("dtypes:", df_clean.dtypes)

    with (ro.default_converter + pandas2ri.converter).context():
        #Python ->R
        Y_r = conversion.py2rpy(df_clean)

        ro.globalenv["Y"] = Y_r
        ro.globalenv["node_type"] = node_type
        ro.globalenv["seed"] = seed
        ro.r('''
        set.seed(seed)
        Y_df <- as.data.frame(Y)

        # Convert to numeric matrix for hc
        Y_mat <- as.matrix(Y_df)
        
        temp <- dagbagM::hc(
          Y = Y_mat,
          nodeType = node_type,
          whiteList = NULL,
          blackList = NULL,
          tol = 1e-6,
          standardize = FALSE,
          maxStep = 1000,
          restart = 10,
          verbose = FALSE
        )
        adj_mat <- temp$adjacency
        col_names <- colnames(Y_mat)
        ''')
        #R -> Python
        adjacency = conversion.rpy2py(ro.r('adj_mat'))
        col_names= list(ro.r('col_names'))

    return np.asarray(adjacency), col_names #list(df_clean.columns) #returns labels.

In [36]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj, nodes, output_path):
    G = nx.DiGraph()
    #add nodes
    G.add_nodes_from(nodes)
    
    #add directed edges, adj[i,j] = 1 => i -> j
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                G.add_edge(src, tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [33]:
import os
from pathlib import Path
#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

#path to put results
output_dir = project_root/"results"/"graphs_DAGBagM"
output_dir.mkdir(parents=True,exist_ok=True)

In [37]:
csv_files = sorted(cp_root.rglob("*.csv"))

for csv_path in csv_files:
    #relative path
    rel = csv_path.relative_to(cp_root)
    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        # Run DAGBagM on this dataset
        adj,nodes = run_dagbagm(df, seed=1)

        #Output schema scenario__file__DAGBagM.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__DAGBagM.png"
        out_path = output_dir / out_name

        # Save PNG
        draw_graph(adj, nodes, out_path)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

Processing: berkson_paradox/.ipynb_checkpoints/admission_bias-checkpoint.csv
Columns used in dagbagM: ['TestHigh', 'ExtraHigh']
dtypes: TestHigh     int64
ExtraHigh    int64
dtype: object
  Saved graph to results/graphs_DAGBagM/berkson_paradox__admission_bias-checkpoint__DAGBagM.png
Processing: berkson_paradox/.ipynb_checkpoints/loan_approval_bias-checkpoint.csv
Columns used in dagbagM: ['Creditworthy', 'Employed']
dtypes: Creditworthy    int64
Employed        int64
dtype: object
  Saved graph to results/graphs_DAGBagM/berkson_paradox__loan_approval_bias-checkpoint__DAGBagM.png
Processing: berkson_paradox/.ipynb_checkpoints/movie_success_bias-checkpoint.csv
Columns used in dagbagM: ['HighBudget', 'StarPower']
dtypes: HighBudget    int64
StarPower     int64
dtype: object
  Saved graph to results/graphs_DAGBagM/berkson_paradox__movie_success_bias-checkpoint__DAGBagM.png
Processing: berkson_paradox/admission_bias.csv
Columns used in dagbagM: ['TestHigh', 'ExtraHigh']
dtypes: TestHigh     

R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  


  Saved graph to results/graphs_DAGBagM/casual_effect__device_failure_data__DAGBagM.png
Processing: casual_effect/machine_maintenance_data.csv
Columns used in dagbagM: ['equipment_age', 'initial_wear', 'maintenance_round1', 'wear_after1', 'maintenance_round2', 'breakdown', 'plant_id', 'noise_var']
dtypes: equipment_age           int64
initial_wear            int64
maintenance_round1      int64
wear_after1             int64
maintenance_round2      int64
breakdown               int64
plant_id               object
noise_var             float64
dtype: object
  ERROR on casual_effect/machine_maintenance_data.csv: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].

Processing: casual_effect/marketing_offer_data.csv
Columns used in dagbagM: ['tenure_months', 'base_engagement', 'initial_offer', 'followup_engagement', 'special_offer', 'purchase', 'region', 'noise_factor']
dtypes: tenure_mont

R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  


  Saved graph to results/graphs_DAGBagM/mediation_outcome_confounder__nutrition_program_study__DAGBagM.png
Processing: mediation_outcome_confounder/rehabilitation_program_study.csv
Columns used in dagbagM: ['received_rehab_program', 'initial_injury_severity', 'injury_type', 'baseline_mobility_score', 'pain_tolerance_level', 'weekly_therapy_hours', 'final_mobility_score', 'unrelated_random_variable']
dtypes: received_rehab_program         int64
initial_injury_severity      float64
injury_type                   object
baseline_mobility_score      float64
pain_tolerance_level         float64
weekly_therapy_hours         float64
final_mobility_score         float64
unrelated_random_variable    float64
dtype: object
  ERROR on mediation_outcome_confounder/rehabilitation_program_study.csv: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].

Processing: mediation_outcome_confounder/student

...

  Saved graph to results/graphs_DAGBagM/necessity_sufficiency__network_health__DAGBagM.png
Processing: necessity_sufficiency/patient_recovery.csv
Columns used in dagbagM: ['treatment_dosage', 'inflammation_level', 'activity_level', 'recovery_score']
dtypes: treatment_dosage      float64
inflammation_level    float64
activity_level        float64
recovery_score        float64
dtype: object
  Saved graph to results/graphs_DAGBagM/necessity_sufficiency__patient_recovery__DAGBagM.png
Processing: necessity_sufficiency/stress_sem.csv
Columns used in dagbagM: ['Temperature', 'Pressure', 'Vibration_level', 'Stress_Severity']
dtypes: Temperature        float64
Pressure           float64
Vibration_level    float64
Stress_Severity    float64
dtype: object
  Saved graph to results/graphs_DAGBagM/necessity_sufficiency__stress_sem__DAGBagM.png
Processing: observational_vs_experimental_reasoning/diet_adherence_study.csv
Columns used in dagbagM: ['diet_adherence', 'weight_loss_success', 'exercise_regu

R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(),

Columns used in dagbagM: ['meditation_use', 'stress_reduction', 'stress_high', 'age_old', 'smartphone_use', 'noise_level', 'sleep_habit']
dtypes: meditation_use       int64
stress_reduction     int64
stress_high          int64
age_old              int64
smartphone_use       int64
noise_level          int64
sleep_habit         object
dtype: object
  ERROR on observational_vs_experimental_reasoning/meditation_app_study.csv: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].

Processing: observational_vs_experimental_reasoning/observational_sleep_study.csv
Columns used in dagbagM: ['supplement', 'sleep_quality', 'exercise_regularly', 'age_old', 'coffee_drinker', 'watch_tv_late', 'income']
dtypes: supplement             int64
sleep_quality          int64
exercise_regularly     int64
age_old                int64
coffee_drinker         int64
watch_tv_late          int64
income            

R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  
R callback write-console: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].
  


  Saved graph to results/graphs_DAGBagM/sequential_mediator__sequential_mediation_variant4__DAGBagM.png
Processing: simpson_paradox/medicine.csv
Columns used in dagbagM: ['drug', 'age', 'recover', 'SES', 'hospital_rating', 'random_noise']
dtypes: drug                object
age                 object
recover              int64
SES                 object
hospital_rating    float64
random_noise       float64
dtype: object
  ERROR on simpson_paradox/medicine.csv: Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  Not compatible with requested type: [type=character; target=double].

Processing: simpson_paradox/medicine_variant1.csv
Columns used in dagbagM: ['Drug', 'Age', 'Recovery', 'SES', 'HospitalRating', 'RandomNoise']
dtypes: Drug               object
Age                object
Recovery            int64
SES                object
HospitalRating    float64
RandomNoise       float64
dtype: object
  ERROR on simpson_paradox/medicine_variant1.csv: Error in